<a href="https://colab.research.google.com/github/basharxwar/Online-Retail/blob/main/Online_Retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
df=pd.read_excel("/content/drive/MyDrive/Online Retail.xlsx")
df_raw = df.copy()      # original copy, we never change it
print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [6]:
df.info()
print()
print(df.isna().sum())
print()
print("duplicate rows:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

duplicate rows: 5268


In [8]:
dups = df[df.duplicated(keep=False)]
dups.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [7]:
no_desc = (df[df["Description"].isna()]["UnitPrice"] == 0).sum()
no_desc

np.int64(1454)

In [9]:
df[df.duplicated()]["Quantity"].value_counts().head()

,count
Quantity,
1,3732
2,584
6,212
12,205
3,131


In [10]:
(df["UnitPrice"] == 0).sum()

np.int64(2515)

In [11]:
(df["UnitPrice"] < 0).sum()

np.int64(2)

In [12]:
rows_before = len(df)
# Decision: remove exact duplicate rows, keep the first copy
df = df.drop_duplicates()
print("rows removed:", rows_before - len(df))

rows removed: 5268


In [13]:
rows_before = len(df)
# Decision: remove rows with UnitPrice == 0 (nothing was paid, so they are not sales)
df = df[df["UnitPrice"] != 0]
print("rows removed:", rows_before - len(df))
print("Description still missing:", df["Description"].isna().sum())

rows removed: 2510
Description still missing: 0


In [14]:
df[df["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [15]:
neg = df[df["Quantity"] < 0]
print("negative quantities:", len(neg))
print("of them in invoices starting with C:", neg["InvoiceNo"].astype(str).str.startswith("C").sum())

negative quantities: 9251
of them in invoices starting with C: 9251


In [16]:
rows_before = len(df)
# Decision: remove rows with a negative UnitPrice (accounting entries "Adjust bad debt", not products)
df = df[df["UnitPrice"] >= 0]
print("rows removed:", rows_before - len(df))

rows removed: 2


In [17]:
# Decision: keep negative quantities. All 9,251 are in invoices starting with C (returns), so they are real events
df["is_cancellation"] = df["InvoiceNo"].astype(str).str.startswith("C")
print(df["is_cancellation"].sum())

9251


In [18]:
df[df["Quantity"].abs() > 10000]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancellation
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,False
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,True
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,False
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,True


In [19]:
# Decision: do not fill the missing CustomerID (we cannot invent customer numbers); keep these rows because the sales are real
# Decision: CustomerID, InvoiceNo and StockCode are labels, not numbers to calculate with -> text
df["CustomerID"] = df["CustomerID"].astype("Int64").astype("string")
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)
print(df.dtypes)
print("missing CustomerID:", df["CustomerID"].isna().sum())
print("rows now:", len(df))

InvoiceNo                  object
StockCode                  object
Description                object
Quantity                    int64
InvoiceDate        datetime64[ns]
UnitPrice                 float64
CustomerID         string[python]
Country                    object
is_cancellation              bool
dtype: object
missing CustomerID: 132565
rows now: 534129


In [20]:
df.nlargest(8, "UnitPrice")[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
222681,C556445,M,Manual,-1,38970.00
524602,C580605,AMAZONFEE,AMAZON FEE,-1,17836.46
43702,C540117,AMAZONFEE,AMAZON FEE,-1,16888.02
43703,C540118,AMAZONFEE,AMAZON FEE,-1,16453.71
15016,C537630,AMAZONFEE,AMAZON FEE,-1,13541.33
15017,537632,AMAZONFEE,AMAZON FEE,1,13541.33
16356,C537651,AMAZONFEE,AMAZON FEE,-1,13541.33
16232,C537644,AMAZONFEE,AMAZON FEE,-1,13474.79


In [21]:
before_after = pd.DataFrame({
    "before": [len(df_raw), df_raw.isna().sum().sum(), df_raw["Description"].isna().sum(),
               df_raw["CustomerID"].isna().sum(), df_raw.duplicated().sum(),
               (df_raw["UnitPrice"] <= 0).sum(), (df_raw["Quantity"] < 0).sum()],
    "after":  [len(df), df.isna().sum().sum(), df["Description"].isna().sum(),
               df["CustomerID"].isna().sum(), df.duplicated().sum(),
               (df["UnitPrice"] <= 0).sum(), (df["Quantity"] < 0).sum()],
}, index=["rows", "missing cells (all columns)", "missing Description", "missing CustomerID",
          "duplicate rows", "UnitPrice <= 0", "negative Quantity (returns, kept)"])
before_after

,before,after
rows,541909,534129
missing cells (all columns),136534,132565
missing Description,1454,0
missing CustomerID,135080,132565
duplicate rows,5268,0
UnitPrice <= 0,2517,0
"negative Quantity (returns, kept)",10624,9251


In [22]:
not_product = ~df["StockCode"].str.match(r"^\d{5}")
print("rows that do not have a 5-digit product code:", not_product.sum())
df[not_product]["StockCode"].value_counts().head(15)

rows that do not have a 5-digit product code: 2957


,count
StockCode,
POST,1252
DOT,707
M,560
C2,143
D,77
S,62
BANK CHARGES,37
AMAZONFEE,34
CRUK,16


In [23]:
not_product = ~df["StockCode"].str.match(r"^\d{5}")
top_codes = df[not_product]["StockCode"].value_counts().head(15).index
df[df["StockCode"].isin(top_codes)].groupby("StockCode")["Description"].agg(["size", "first"])

,size,first
StockCode,,
AMAZONFEE,34,AMAZON FEE
BANK CHARGES,37,Bank Charges
C2,143,CARRIAGE
CRUK,16,CRUK Commission
D,77,Discount
DCGS0003,4,BOXED GLASS ASHTRAY
DCGSSBOY,11,BOYS PARTY BAG
DCGSSGIRL,13,GIRLS PARTY BAG
DOT,707,DOTCOM POSTAGE


In [24]:
print("BEFORE")
print(df_raw[["Quantity", "UnitPrice"]].describe().round(2))
print()
print("AFTER")
print(df[["Quantity", "UnitPrice"]].describe().round(2))

BEFORE
        Quantity  UnitPrice
count  541909.00  541909.00
mean        9.55       4.61
std       218.08      96.76
min    -80995.00  -11062.06
25%         1.00       1.25
50%         3.00       2.08
75%        10.00       4.13
max     80995.00   38970.00

AFTER
        Quantity  UnitPrice
count  534129.00  534129.00
mean        9.92       4.70
std       216.45      95.08
min    -80995.00       0.00
25%         1.00       1.25
50%         3.00       2.10
75%        10.00       4.13
max     80995.00   38970.00


In [25]:
rows_before = len(df)
fee_codes = ["POST", "DOT", "M", "C2", "D", "S", "BANK CHARGES", "AMAZONFEE", "CRUK"]
# Decision 6: remove rows that are shipping, discounts, fees or manual entries, not products
mask = df["StockCode"].isin(fee_codes)
removed_fees = df[mask]          # kept aside in case we need them
df = df[~mask]
print("rows removed:", rows_before - len(df))

not_product = ~df["StockCode"].str.match(r"^\d{5}")
print(df[not_product]["StockCode"].value_counts())

rows removed: 2888
StockCode
DCGSSGIRL       13
DCGSSBOY        11
gift_0001_20     9
gift_0001_10     8
gift_0001_30     7
gift_0001_50     4
DCGS0003         4
PADS             3
gift_0001_40     3
DCGS0076         2
m                1
DCGS0070         1
DCGS0069         1
DCGS0004         1
B                1
Name: count, dtype: int64


In [26]:
print("BEFORE")
print(df_raw[["Quantity", "UnitPrice"]].describe().round(2))
print()
print("AFTER")
print(df[["Quantity", "UnitPrice"]].describe().round(2))

BEFORE
        Quantity  UnitPrice
count  541909.00  541909.00
mean        9.55       4.61
std       218.08      96.76
min    -80995.00  -11062.06
25%         1.00       1.25
50%         3.00       2.08
75%        10.00       4.13
max     80995.00   38970.00

AFTER
        Quantity  UnitPrice
count  531241.00  531241.00
mean        9.96       3.31
std       217.00      15.83
min    -80995.00       0.00
25%         1.00       1.25
50%         3.00       2.08
75%        10.00       4.13
max     80995.00   11062.06


In [27]:
path = "/content/drive/MyDrive/online_retail_clean.csv.gz"
df.to_csv(path, index=False, compression="gzip")
import os
print("file size (MB):", round(os.path.getsize(path) / 1e6, 2))

file size (MB): 7.63


In [29]:
pd.read_csv(path, dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"}, parse_dates=["InvoiceDate"])

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancellation
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
531236,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,False
531237,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,False
531238,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,False
531239,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,False


In [30]:
df.nlargest(5, "UnitPrice")[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
299982,A563185,B,Adjust bad debt,1,11062.06
222680,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50
222682,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,649.50
4989,536835,22655,VINTAGE RED KITCHEN CABINET,1,295.00
32484,539080,22655,VINTAGE RED KITCHEN CABINET,1,295.00


In [31]:
df[~df["StockCode"].str.match(r"^\d{5}")]["StockCode"].value_counts()

,count
StockCode,
DCGSSGIRL,13
DCGSSBOY,11
gift_0001_20,9
gift_0001_10,8
gift_0001_30,7
gift_0001_50,4
DCGS0003,4
PADS,3
gift_0001_40,3


In [32]:
df.nsmallest(5, "UnitPrice")[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
157195,550193,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001
279045,561226,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001
359871,568200,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.030
217060,555876,16045,POPART WOODEN PENCILS ASST,100,0.040


In [33]:
rows_before = len(df)
# Decision 7: remove the accounting row "Adjust bad debt" (StockCode B) that was left over, it is not a product
df = df[df["StockCode"] != "B"]
print("rows removed:", rows_before - len(df))

rows removed: 1


In [34]:
df[df["StockCode"] == "m"]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancellation
40383,539736,m,Manual,1,2010-12-21 15:18:00,2.55,<NA>,United Kingdom,False


In [35]:
rows_before = len(df)
# Decision 8: remove leftover manual entries: the lowercase code "m", and invoice 556444 (60 picnic baskets = 38,970), which a manual entry cancelled
df = df[(df["StockCode"] != "m") & (df["InvoiceNo"] != "556444")]
print("rows removed:", rows_before - len(df))

rows removed: 2


In [36]:
print("rows:", len(df))                          # expected 531238
print("UnitPrice max:", df["UnitPrice"].max())   # expected 649.5
print("duplicates:", df.duplicated().sum())      # expected 0

rows: 531238
UnitPrice max: 649.5
duplicates: 0


In [37]:
before_after = pd.DataFrame({
    "before": [len(df_raw), df_raw.isna().sum().sum(), df_raw["Description"].isna().sum(),
               df_raw["CustomerID"].isna().sum(), df_raw.duplicated().sum(),
               (df_raw["UnitPrice"] <= 0).sum(), (df_raw["Quantity"] < 0).sum(),
               df_raw["UnitPrice"].max(), round(df_raw["UnitPrice"].std(), 2)],
    "after":  [len(df), df.isna().sum().sum(), df["Description"].isna().sum(),
               df["CustomerID"].isna().sum(), df.duplicated().sum(),
               (df["UnitPrice"] <= 0).sum(), (df["Quantity"] < 0).sum(),
               df["UnitPrice"].max(), round(df["UnitPrice"].std(), 2)],
}, index=["rows", "missing cells (all columns)", "missing Description", "missing CustomerID",
          "duplicate rows", "UnitPrice <= 0", "negative Quantity (returns, kept)",
          "UnitPrice max", "UnitPrice std"])
before_after

,before,after
rows,541909.00,531238.00
missing cells (all columns),136534.00,131580.00
missing Description,1454.00,0.00
missing CustomerID,135080.00,131580.00
duplicate rows,5268.00,0.00
UnitPrice <= 0,2517.00,0.00
"negative Quantity (returns, kept)",10624.00,8668.00
UnitPrice max,38970.00,649.50
UnitPrice std,96.76,4.43


In [38]:
print("BEFORE")
print(df_raw[["Quantity", "UnitPrice"]].describe().round(2))
print()
print("AFTER")
print(df[["Quantity", "UnitPrice"]].describe().round(2))

BEFORE
        Quantity  UnitPrice
count  541909.00  541909.00
mean        9.55       4.61
std       218.08      96.76
min    -80995.00  -11062.06
25%         1.00       1.25
50%         3.00       2.08
75%        10.00       4.13
max     80995.00   38970.00

AFTER
        Quantity  UnitPrice
count  531238.00  531238.00
mean        9.96       3.29
std       217.00       4.43
min    -80995.00       0.00
25%         1.00       1.25
50%         3.00       2.08
75%        10.00       4.13
max     80995.00     649.50


In [39]:
import os
path = "/content/drive/MyDrive/online_retail_clean.csv.gz"
df.to_csv(path, index=False, compression="gzip")
print("file size (MB):", round(os.path.getsize(path) / 1e6, 2))

check = pd.read_csv(path, dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
                    parse_dates=["InvoiceDate"])
print("saved file has the same shape:", check.shape == df.shape, check.shape)

file size (MB): 7.63
saved file has the same shape: True (531238, 9)


In [40]:
   print(df["is_cancellation"].sum())      # expected 8668

8668
